# Appendix 4 - Fourier Series, Transforms, and FFTs in Python

**How do FS, FT, DTFT, DFT, and FFT fit together?**

This is a general signal-analysis appendix using **spatial signals** throughout. Position $x$ is measured in nm and spatial frequency $f_x$ in cycles/nm. We distinguish the mathematical objects, compare an analytical Fourier transform with a scaled FFT, and then work through the NumPy calls.

Run the code cells in order. Only NumPy and Matplotlib are needed. Signals are dimensionless in the examples; $i=\sqrt{-1}$.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## The transform family: different questions, related representations

| Name | Starting description | Result |
| --- | --- | --- |
| **FS: Fourier series** | A continuous periodic signal, such as $u(x+d)=u(x)$ | Discrete harmonic coefficients $c_m$ |
| **FT: Fourier transform** | A continuous signal, typically aperiodic | A function of continuous frequency $U(f_x)$ |
| **DTFT: discrete-time Fourier transform** | A sequence $u[j]$ indexed by integers, including spatial samples | A continuous, $2\pi$-periodic function of normalized frequency $\theta$ |
| **DFT: discrete Fourier transform** | A finite record of $N$ samples | $N$ frequency-bin values $F_k$ |
| **FFT: fast Fourier transform** | The same finite record | An efficient algorithm computing its DFT |

The conventional name **DTFT** contains "time", but its defining sum applies equally to spatial samples. A discrete input coordinate does not imply a discrete frequency coordinate. FFT is an algorithm, not a different transform from the DFT. Fourier series also exist for discrete periodic sequences; the FS row above specifies the continuous-coordinate case.

### Fourier series versus Fourier transform

For a spatial signal with period $d$, let $f_0=1/d$. Fourier series writes

$$u(x)=\sum_{m=-\infty}^{\infty}c_m e^{i2\pi m f_0x},\qquad
c_m=\frac{1}{d}\int_{x_a}^{x_a+d}u(x)e^{-i2\pi m f_0x}\,dx.$$

The harmonics occur at $f_x=m/d$ in cycles/nm. In angular spatial frequency, they occur at $k_x=mG$, where $G=2\pi/d$ in rad/nm. This is the connection between a physical spatial period and its harmonic spacing.

For a real signal, the equivalent sine-cosine form is

$$u(x)=a_0+\sum_{m=1}^{\infty}\left[a_m\cos(2\pi m f_0x)+b_m\sin(2\pi m f_0x)\right],$$
$$c_0=a_0,\qquad c_m=(a_m-i b_m)/2,\qquad c_{-m}=c_m^*.$$

Here $a_0$ is the mean value; another common convention writes the mean as $a_0/2$. Computing the coefficients is **analysis**; reconstructing the signal from them is **synthesis**. The real and complex forms encode the same signal.

For an aperiodic spatial signal, use the transform pair

$$U(f_x)=\int_{-\infty}^{\infty}u(x)e^{-i2\pi f_xx}\,dx,\qquad
u(x)=\int_{-\infty}^{\infty}U(f_x)e^{i2\pi f_xx}\,df_x.$$

Frequency is continuous, so reconstruction uses an integral. We use cycles/nm; angular spatial frequency is $k_x=2\pi f_x$ in rad/nm. Other Fourier conventions move factors of $2\pi$ between these equations.

A periodic signal can also have an FT interpreted as spectral impulses:

$$U(f_x)=\sum_m c_m\,\delta(f_x-mf_0).$$

Thus Fourier-series coefficients and a continuous FT are related, but a line weight $c_m$ is not the height of an ordinary continuous spectral curve. These formulas require the usual convergence assumptions or a generalized-function interpretation.

### One rectangular feature versus its periodic repetition

Let $p(x)=A\,\operatorname{rect}[(x-x_c)/a]$ be a single feature of width $a$ nm. With $\operatorname{sinc}(v)=\sin(\pi v)/(\pi v)$,

$$P(f_x)=Aa\,\operatorname{sinc}(a f_x)e^{-i2\pi f_xx_c}.$$

Repeat that feature every $d>a$. Integration over one period gives the Fourier-series coefficients

$$\boxed{c_m=\frac{P(m/d)}{d}
=A\frac{a}{d}\operatorname{sinc}\!\left(\frac{ma}{d}\right)e^{-i2\pi mx_c/d}.}$$

The continuous transform of one feature supplies the line weights of its periodic repetition, evaluated at harmonic frequencies and divided by the period. The plots use separate vertical scales: $P$ has signal-times-nm units, while $c_m$ has signal units. A stem marks a line's weight; it is not the height of a Dirac delta.

**Optics notation:** the generic series coefficient $c_m$ here belongs to the chosen signal. For the course's optical fields, write $t_m$ when that signal is a mask transmission $t$, and $C_m$ when it is the exit field $U_{\mathrm{exit}}$. See the [shared convention](0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


In [ ]:
feature_width_nm, feature_period_nm, feature_centre_nm = 200.0, 1000.0, 80.0
feature_amplitude = 1.0
feature_fx = np.linspace(-0.02, 0.02, 2001)
feature_m = np.arange(-20, 21)

def feature_ft(fx):
    return (feature_amplitude * feature_width_nm * np.sinc(feature_width_nm * fx)
            * np.exp(-2j * np.pi * fx * feature_centre_nm))

feature_coefficients = feature_ft(feature_m / feature_period_nm) / feature_period_nm
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)
axes[0].plot(feature_fx, np.abs(feature_ft(feature_fx)))
axes[0].set(xlabel='Spatial frequency fx (cycles/nm)', ylabel='|P(fx)| (nm)',
            title='One feature: continuous FT')
axes[1].plot(feature_fx, np.abs(feature_ft(feature_fx)) / feature_period_nm,
             color='gray', linestyle='--', label='|P(fx)| / period envelope')
axes[1].stem(feature_m / feature_period_nm, np.abs(feature_coefficients), basefmt=' ')
axes[1].set(xlabel='Harmonic spatial frequency m/d (cycles/nm)', ylabel='|c_m| (dimensionless)',
            title='Periodic features: discrete FS weights')
axes[1].legend(fontsize=8)
plt.show()
plt.close(fig)

### DTFT, DFT, and FFT

The DTFT of a sequence is

$$U_d(\theta)=\sum_{j=-\infty}^{\infty}u[j]e^{-i\theta j},\qquad
U_d(\theta+2\pi)=U_d(\theta).$$

Here $\theta$ is in radians per sample. For spatial samples spaced by $\Delta x$, $\theta=2\pi f_x\Delta x$; the periodicity in physical spatial frequency is $1/\Delta x$.

For a finite record $u[0],\ldots,u[N-1]$, temporarily set the samples outside the record to zero. Its DTFT is a finite sum that we can evaluate at any frequency. The DFT samples that function:

$$F_k=\sum_{j=0}^{N-1}u[j]e^{-i2\pi jk/N}
=U_{d,\mathrm{record}}(2\pi k/N),\qquad k=0,\ldots,N-1.$$

The inverse DFT also lets us interpret the same record as one period of a discrete periodic signal, whose series coefficients are $F_k/N$. Windowing a longer sequence first changes its DTFT; the DFT samples the transform of the record actually supplied.

The next plot evaluates a four-sample spatial record's DTFT directly, then overlays its DFT samples. Padding that record with zeros gives more samples of the same DTFT curve; it adds no measured signal values. The plot uses raw DFT values, without dividing by either FFT length.

In [ ]:
record = np.array([1.0, 0.6, 0.2, 0.4])
sample_index = np.arange(record.size)
theta_dense = np.linspace(-np.pi, np.pi, 2001)
dtft_dense = np.exp(-1j * theta_dense[:, None] * sample_index) @ record

fig, ax = plt.subplots(figsize=(8.5, 3.4), constrained_layout=True)
ax.plot(theta_dense, np.abs(dtft_dense), color='black', label='Continuous DTFT of the record')
for length, marker in ((16, 'x'), (4, 'o')):
    theta_bins = 2 * np.pi * np.fft.fftfreq(length)
    dft_values = np.fft.fft(record, n=length)
    direct_values = np.exp(-1j * theta_bins[:, None] * sample_index) @ record
    np.testing.assert_allclose(dft_values, direct_values, rtol=0, atol=1e-12)
    ax.scatter(theta_bins, np.abs(dft_values), marker=marker, s=55,
               label=f'{length}-point FFT: DFT samples')
ax.set(xlabel='Normalized spatial frequency theta (rad/sample)', ylabel='Transform magnitude',
       xlim=(-np.pi, np.pi), title='A continuous DTFT, sampled by the DFT')
ax.legend(fontsize=9)
plt.show()
plt.close(fig)

## Analytical FT to FFT: where the scale factor comes from

An analytical transform evaluates a continuous integral. An FFT evaluates a finite sum. Sample at $x_j=x_a+j\Delta x$ and let $L=N\Delta x$. At the DFT spatial frequencies $f_{x,k}=k/L$, with signed bin indices as returned by `fftfreq`, the integral over the sampled window is approximated by

$$U(f_{x,k})\approx\Delta x\sum_{j=0}^{N-1}u(x_j)e^{-i2\pi f_{x,k}x_j}
=\boxed{\Delta x\,e^{-i2\pi f_{x,k}x_a}\,F_k}.$$

The factor $\Delta x$ comes from the integration measure. The origin factor accounts for $x_a\ne0$. Sampling and truncation must be adequate before this finite-window sum approximates the full FT. By comparison, $e^{-i2\pi f_{x,k}x_a}F_k/N$ estimates Fourier-series coefficients for the periodic extension of that window. The two normalizations differ by $L$.

| Input coordinate | Transform coordinate | Continuous-transform value units |
| --- | --- | --- |
| $x$ in nm | $f_x$ in cycles/nm | Signal units multiplied by nm |
| $(x,y)$ in (nm, nm) | $(f_x,f_y)$ in (cycles/nm, cycles/nm) | Signal units multiplied by nm$^2$ |
| $t$ in s, if time were used instead | $f$ in Hz | Signal units multiplied by s |

These are coordinate units, separate from signal-amplitude units. A two-dimensional integral needs the product of the two sample spacings, $\Delta x\Delta y$.

### A transform we can calculate analytically

Choose a shifted spatial Gaussian, $g(x)=\exp[-((x-x_c)/w)^2]$. Substitution of $(x-x_c)/w$ into the Fourier integral gives

$$\boxed{G(f_x)=w\sqrt{\pi}\,e^{-(\pi w f_x)^2}e^{-i2\pi f_xx_c}.}$$

The spatial shift changes spectral phase without changing spectral magnitude. This lets us test the complex transform, including its origin correction, rather than comparing only magnitudes. Our Gaussian is negligible at both edges of the chosen spatial window.

In [ ]:
gaussian_N = 1024
gaussian_L_nm = 1600.0
gaussian_dx_nm = gaussian_L_nm / gaussian_N
gaussian_start_nm = -gaussian_L_nm / 2
gaussian_x_nm = gaussian_start_nm + np.arange(gaussian_N) * gaussian_dx_nm
width_nm, centre_nm = 80.0, 75.0
gaussian = np.exp(-((gaussian_x_nm - centre_nm) / width_nm)**2)
gaussian_fx = np.fft.fftfreq(gaussian_N, d=gaussian_dx_nm)

def gaussian_ft(fx):
    return (width_nm * np.sqrt(np.pi) * np.exp(-(np.pi * width_nm * fx)**2)
            * np.exp(-2j * np.pi * fx * centre_nm))

gaussian_raw = np.fft.fft(gaussian)
origin_factor = np.exp(-2j * np.pi * gaussian_fx * gaussian_start_nm)
gaussian_numerical = gaussian_dx_nm * origin_factor * gaussian_raw
gaussian_analytical = gaussian_ft(gaussian_fx)
peak_scale = float(np.max(np.abs(gaussian_analytical)))
complex_error = float(np.max(np.abs(gaussian_numerical - gaussian_analytical)) / peak_scale)
uncorrected_error = float(np.max(np.abs(gaussian_dx_nm * gaussian_raw - gaussian_analytical)) / peak_scale)

print(f'Max complex error / analytical peak: {complex_error:.2e}')
print(f'Without the origin factor:           {uncorrected_error:.2e}')
print(f'G(0): analytical = {gaussian_ft(0):.6f} nm; FFT approximation = {gaussian_numerical[0]:.6f} nm')
np.testing.assert_allclose(gaussian_numerical, gaussian_analytical, rtol=0, atol=1e-10)
gaussian_back = np.fft.ifft(gaussian_numerical / (gaussian_dx_nm * origin_factor))
np.testing.assert_allclose(gaussian_back, gaussian, rtol=0, atol=1e-12)

In [ ]:
fx_curve = np.linspace(-0.015, 0.015, 1201)
G_curve = gaussian_ft(fx_curve)
near_peak = np.abs(gaussian_fx) <= 0.015
fx_bins = gaussian_fx[near_peak]
G_bins = gaussian_numerical[near_peak]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
axes[0].plot(gaussian_x_nm, gaussian)
axes[0].set(xlim=(-300, 400), xlabel='Position x (nm)', ylabel='g(x) (dimensionless)',
            title='Shifted spatial Gaussian')
axes[1].plot(fx_curve, np.abs(G_curve), color='black', label='Analytical FT')
axes[1].scatter(fx_bins, np.abs(G_bins), s=15, label='Scaled FFT')
axes[1].set(xlabel='Spatial frequency fx (cycles/nm)', ylabel='|G(fx)| (nm)',
            title='Spectral magnitude')
axes[1].legend(fontsize=8)
axes[2].plot(fx_curve, G_curve.real, color='tab:blue', label='Analytical Re G')
axes[2].plot(fx_curve, G_curve.imag, color='tab:orange', label='Analytical Im G')
axes[2].scatter(fx_bins, G_bins.real, color='tab:blue', s=12)
axes[2].scatter(fx_bins, G_bins.imag, color='tab:orange', marker='x', s=18)
axes[2].set(xlabel='Spatial frequency fx (cycles/nm)', ylabel='Transform components (nm)',
            title='Complex agreement tests phase')
axes[2].legend(fontsize=8)
plt.show()
plt.close(fig)

Here the FT at zero frequency is the area under the Gaussian, $w\sqrt{\pi}$. The agreement is for this well-resolved, effectively untruncated example. An FFT/inverse-FFT round trip alone could not establish it: an aliased or heavily truncated record still has an invertible DFT.

The Gaussian's raw FFT, its normalized Fourier-series coefficients, and its integral-transform approximation are three differently scaled descriptions. The following practical examples use `F/N` for coefficients and keep raw `F` for inverse transforms.

## 1. Make spatial samples and keep their spacing

Start with 128 samples across 1000 nm. The last sample is at $1000-\Delta x$ nm, so we do not repeat the endpoint at 1000 nm.

Our signal contains a constant offset and two spatial cosines. Their frequencies are $5/L$ and $12/L$ cycles/nm, so they complete whole cycles within the window. Later, try changing `5.0` to `5.5`: the peak spreads across bins because the window no longer contains a whole number of cycles. Sampling and window effects are developed in 9.1.

In [ ]:
N = 128
L_nm = 1000.0
dx_nm = L_nm / N
x_nm = np.arange(N) * dx_nm
f1 = 5.0 / L_nm  # Cycles/nm: five cycles over the window.
f2 = 12.0 / L_nm
u = (0.3
     + 2.0 * np.cos(2 * np.pi * f1 * x_nm + 0.4)
     + 0.5 * np.cos(2 * np.pi * f2 * x_nm - 0.7))

print(f'{N} samples; spacing {dx_nm:.6f} nm; window length N*dx = {L_nm:.3f} nm')

## 2. Transform, label the spatial frequencies, and invert

`fft` receives the sample values. It does **not** know `dx_nm`. Use `fftfreq(N, d=dx_nm)` to attach a physical spatial frequency to every output element. With nm as the spacing unit, frequencies are in cycles/nm.

The raw spectrum starts with zero frequency, then positive frequencies, then negative frequencies. Array position is not itself a frequency. Multiply by $2\pi$ if angular spatial frequency in rad/nm is required.

NumPy's default forward transform is unscaled; the inverse includes $1/N$. Keep the raw complex spectrum for reconstruction. Tiny round-trip differences are floating-point error.

In [ ]:
F = np.fft.fft(u)
fx = np.fft.fftfreq(N, d=dx_nm)
u_back = np.fft.ifft(F)

print('First frequency bins:', fx[:8], 'cycles/nm')
print('Last frequency bins: ', fx[-5:], 'cycles/nm')
print(f'Maximum round-trip error: {np.max(np.abs(u_back - u)):.2e}')
np.testing.assert_allclose(u_back, u, rtol=0, atol=1e-12)

## 3. Read amplitude and phase

For these bin-aligned components, `F/N` gives the complex coefficients of the discrete Fourier series. Use `abs` for their magnitudes and `angle` for phases in radians. Phase at a numerically zero coefficient has no useful meaning.

A real cosine of amplitude $A$ contributes two coefficients, one at each signed frequency, each of magnitude $A/2$. The constant term and the even-length Nyquist bin are exceptions to this two-distinct-bin rule. Our cosine at 0.005 cycles/nm therefore produces magnitudes of 1 at both +0.005 and -0.005 cycles/nm.

This is a **two-sided coefficient spectrum**. It is not a calibrated power spectral density.

In [ ]:
C = F / N  # Keep F unchanged for np.fft.ifft(F).
print('Spatial frequency (cycles/nm)   |coefficient|   phase (rad)')
for wanted in (0.0, f1, -f1, f2, -f2):
    k = int(np.argmin(np.abs(fx - wanted)))
    print(f'{fx[k]:+18.5f}              {abs(C[k]):.4f}         {np.angle(C[k]):+.4f}')

## 4. Put zero frequency in the middle for plotting

`fftshift` rearranges array entries. Shift the frequency axis and spectrum together. This changes their display order, not the transform's normalization or physical meaning.

Undo the rearrangement with `ifftshift` before calling `ifft`. Always use the inverse shift, including for odd-length arrays.

In [ ]:
fx_centered = np.fft.fftshift(fx)
F_centered = np.fft.fftshift(F)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)
axes[0].plot(x_nm, u, '.-', markersize=3)
axes[0].set(xlabel='Position x (nm)', ylabel='Signal value', title='Input spatial samples')
axes[1].stem(fx_centered, np.abs(F_centered) / N, basefmt=' ')
axes[1].set(xlim=(-0.016, 0.016), xlabel='Spatial frequency fx (cycles/nm)',
            ylabel='Two-sided coefficient magnitude', title='FFT with physical frequency labels')
plt.show()
plt.close(fig)

u_from_shifted = np.fft.ifft(np.fft.ifftshift(F_centered))
np.testing.assert_allclose(u_from_shifted, u, rtol=0, atol=1e-12)

## 5. Complex input works directly

A complex exponential can have a positive-frequency component without its negative-frequency partner. Keep the complex samples and coefficients throughout. Transforming `abs(u)` or discarding the imaginary part changes the signal.

`rfft` is a compact option for **real input only**. Complex optical fields require `fft` or `fft2`.

In [ ]:
complex_u = np.exp(2j * np.pi * f1 * x_nm + 0.4j)
complex_F = np.fft.fft(complex_u)
positive = int(np.argmin(np.abs(fx - f1)))
negative = int(np.argmin(np.abs(fx + f1)))

print(f'+{f1:.5f} cycles/nm: |coefficient| = {abs(complex_F[positive] / N):.6f}')
print(f'-{f1:.5f} cycles/nm: |coefficient| = {abs(complex_F[negative] / N):.2e}')
np.testing.assert_allclose(np.fft.ifft(complex_F), complex_u, rtol=0, atol=1e-12)

## 6. Two-dimensional spatial signals use `fft2`

For a field shaped `(Ny, Nx)`, rows run along $y$ and columns along $x$. Build a frequency axis for each dimension. Here both sample spacings are 10 nm, so both frequency axes are in cycles/nm.

The two-dimensional coefficient normalization is `F2/(Ny*Nx)`. An approximation to the continuous two-dimensional FT instead uses `dx2_nm*dy2_nm*F2` for this grid starting at $x=y=0$. The inverse uses the raw `F2`. For a stack of fields, specify the intended axes when using FFTs and shifts; `fft2` transforms the last two axes by default.

In [ ]:
Ny, Nx = 48, 64
dx2_nm = dy2_nm = 10.0
x2_nm = np.arange(Nx) * dx2_nm
y2_nm = np.arange(Ny) * dy2_nm
xx, yy = np.meshgrid(x2_nm, y2_nm)
Lx_nm, Ly_nm = Nx * dx2_nm, Ny * dy2_nm
image_data = (np.cos(2 * np.pi * (5 * xx / Lx_nm + 3 * yy / Ly_nm))
              + 0.5 * np.cos(2 * np.pi * (-7 * xx / Lx_nm + 4 * yy / Ly_nm)))

F2 = np.fft.fft2(image_data)
fx2 = np.fft.fftshift(np.fft.fftfreq(Nx, d=dx2_nm))
fy2 = np.fft.fftshift(np.fft.fftfreq(Ny, d=dy2_nm))
spectrum2 = np.abs(np.fft.fftshift(F2, axes=(-2, -1))) / image_data.size

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
axes[0].pcolormesh(x2_nm, y2_nm, image_data, shading='nearest', cmap='gray')
axes[0].set(xlabel='Position x (nm)', ylabel='Position y (nm)', title='Two-dimensional spatial input')
plot = axes[1].pcolormesh(fx2, fy2, spectrum2, shading='nearest', cmap='magma')
axes[1].set(xlim=(-0.02, 0.02), ylim=(-0.02, 0.02), xlabel='fx (cycles/nm)',
            ylabel='fy (cycles/nm)', title='Two pairs of spatial-frequency peaks')
fig.colorbar(plot, ax=axes[1], label='Coefficient magnitude')
plt.show()
plt.close(fig)

image_back = np.fft.ifft2(F2)
print('Input and spectrum shapes:', image_data.shape, F2.shape)
print(f'Maximum 2D round-trip error: {np.max(np.abs(image_back - image_data)):.2e}')
np.testing.assert_allclose(image_back, image_data, rtol=0, atol=1e-12)

## Quick reference

| Task | NumPy call |
| --- | --- |
| Forward / inverse, one dimension | `fft(u)` / `ifft(F)` |
| Forward / inverse, two dimensions | `fft2(u)` / `ifft2(F)` |
| Frequencies matching the raw FFT order | `fftfreq(N, d=spacing)` |
| Centre zero frequency / undo centring | `fftshift(F)` / `ifftshift(F)` |
| Coefficient magnitude / phase | `abs(F/N)` / `angle(F)` for the 1D default normalization |
| Approximate a spatial FT at FFT bins | `dx_nm * exp(-2j*pi*fx*x_start_nm) * fft(u)` |
| Compact real-input transform | `rfft(u)` with `rfftfreq(N, d=spacing)` |

**Try:** change a cosine's amplitude, then its phase. Which FFT values change? Next use a noninteger frequency and observe spectral leakage from the finite record. Aliasing is different: spatial frequencies separated by an integer multiple of the spatial sampling rate 1/dx produce the same complex-exponential samples. Zero-padding does not undo either source of ambiguity.

All working examples here use spatial coordinates. The definitions and NumPy calls apply to any independent coordinate: replacing nm by seconds would replace cycles/nm by Hz. Chapter 9 develops the sampling requirements for optical fields.



**General references:** J. O. Smith, [DTFT and its relation to the DFT](#ref-appendix-4-dtft) [\[1\]](#ref-appendix-4-dtft); A. V. Oppenheim, MIT OpenCourseWare, [discrete-time Fourier series](#ref-appendix-4-mit-dtfs) [\[2\]](#ref-appendix-4-mit-dtfs).

**Official Python references:** [NumPy FFT conventions](#ref-appendix-4-numpy-fft) [\[3\]](#ref-appendix-4-numpy-fft), [frequency bins](#ref-appendix-4-numpy-fftfreq) [\[4\]](#ref-appendix-4-numpy-fftfreq), and [2D transforms](#ref-appendix-4-numpy-fft2) [\[5\]](#ref-appendix-4-numpy-fft2).

[Back to Chapter 9](9_0_From_a_Physical_Model_to_a_Numerical_Experiment.ipynb)

[Appendix 5 — Formulae Sheet](Appendix_5_Formulae_Sheet.ipynb) connects these transforms to the formula chains in every course lesson.

(refs-appendix-4)=

## References

(ref-appendix-4-dtft)=

[1] “Discrete-Time Fourier Transform.” [Online]. Available: [source](https://www.dsprelated.com/freebooks/sasp/Discrete_Time_Fourier_Transform.html).

(ref-appendix-4-mit-dtfs)=

[2] MIT OpenCourseWare, “Lecture 10: Discrete-time Fourier series,” *Signals and Systems*, Spring 2011. [Online]. Available: [source](https://ocw.mit.edu/courses/res-6-007-signals-and-systems-spring-2011/resources/lecture-10-discrete-time-fourier-series/).

(ref-appendix-4-numpy-fft)=

[3] NumPy developers, “Discrete Fourier Transform: Implementation details and Normalization,” NumPy reference documentation. [Online]. Available: [source](https://numpy.org/doc/stable/reference/routines.fft.html).

(ref-appendix-4-numpy-fftfreq)=

[4] NumPy developers, “numpy.fft.fftfreq,” NumPy reference documentation. [Online]. Available: [source](https://numpy.org/doc/stable/reference/generated/numpy.fft.fftfreq.html).

(ref-appendix-4-numpy-fft2)=

[5] NumPy developers, “numpy.fft.fft2,” NumPy reference documentation. [Online]. Available: [source](https://numpy.org/doc/stable/reference/generated/numpy.fft.fft2.html).
